In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,mean_squared_error
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
df = pd.read_csv('airfoil_self_noise.dat', delimiter='\t', names=['frequency', 'attack_angle', 'chord_length', 'free_stream_velocity', 'suction_side_displacement_thickness', 'scaled_sound_pressure'])
df.head()

,frequency,attack_angle,chord_length,free_stream_velocity,suction_side_displacement_thickness,scaled_sound_pressure
0,800,0.0,0.3048,71.3,0.002663,126.201
1,1000,0.0,0.3048,71.3,0.002663,125.201
2,1250,0.0,0.3048,71.3,0.002663,125.951
3,1600,0.0,0.3048,71.3,0.002663,127.591
4,2000,0.0,0.3048,71.3,0.002663,127.461


In [4]:
df.dtypes

frequency                                int64
attack_angle                           float64
chord_length                           float64
free_stream_velocity                   float64
suction_side_displacement_thickness    float64
scaled_sound_pressure                  float64
dtype: object

In [5]:
df.isnull().sum(axis=0)

frequency                              0
attack_angle                           0
chord_length                           0
free_stream_velocity                   0
suction_side_displacement_thickness    0
scaled_sound_pressure                  0
dtype: int64

In [6]:
df.count()

frequency                              1503
attack_angle                           1503
chord_length                           1503
free_stream_velocity                   1503
suction_side_displacement_thickness    1503
scaled_sound_pressure                  1503
dtype: int64

# Linear Regression Model

In [7]:
X = df.drop('scaled_sound_pressure',axis=1)
y = df['scaled_sound_pressure']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2,random_state=42)
lr = LinearRegression()
lr.fit(X_train,y_train)
lr.score(X_test,y_test)

0.5582979754897284

In [8]:
#Normalize the data using MinMaxScaler
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# TensorFlow

In [9]:
tf.random.set_seed(42)

model1 = tf.keras.Sequential(
    [
        tf.keras.layers.Dense(100,activation='relu'),
        tf.keras.layers.Dense(1)
    ]
)

model1.compile(optimizer='adam',loss='mse')

model1.fit(X_train,y_train,batch_size=50,epochs=150,validation_data=(X_test,y_test))

Epoch 1/150
25/25 [==============================] - 2s 20ms/step - loss: 15567.1982 - val_loss: 15444.4521
Epoch 2/150
25/25 [==============================] - 0s 5ms/step - loss: 15408.4277 - val_loss: 15263.1641
Epoch 3/150
25/25 [==============================] - 0s 13ms/step - loss: 15187.0869 - val_loss: 14998.1543
Epoch 4/150
25/25 [==============================] - 0s 9ms/step - loss: 14867.1514 - val_loss: 14629.5352
Epoch 5/150
25/25 [==============================] - 0s 8ms/step - loss: 14445.2197 - val_loss: 14167.9980
Epoch 6/150
25/25 [==============================] - 0s 11ms/step - loss: 13929.2236 - val_loss: 13614.6006
Epoch 7/150
25/25 [==============================] - 0s 11ms/step - loss: 13319.9873 - val_loss: 12974.1592
Epoch 8/150
25/25 [==============================] - 0s 13ms/step - loss: 12625.5986 - val_loss: 12258.1885
Epoch 9/150
25/25 [==============================] - 0s 6ms/step - loss: 11867.1934 - val_loss: 11488.9199
Epoch 10/150
25/25 [============

In [10]:
y_pred = model1.predict(X_test)
print(f'The mean squared error for the test data is {np.round(mean_squared_error(y_test,y_pred),4)}')

10/10 [==============================] - 0s 2ms/step
The mean squared error for the test data is 48.7027


In [11]:
#Change the parameters. Switch from Adam to SGD. Switch epochs to 300.Switch from 100 neurons to 50
tf.random.set_seed(42)

model2 = tf.keras.Sequential(
    [
        tf.keras.layers.Dense(50,activation='relu'),
        tf.keras.layers.Dense(1)
    ]
)

model2.compile(optimizer='sgd',loss='mse')

model2.fit(X_train,y_train,batch_size=50,epochs=300,validation_data=(X_test,y_test))

Epoch 1/300
25/25 [==============================] - 1s 8ms/step - loss: 11540.8975 - val_loss: 7958.3179
Epoch 2/300
25/25 [==============================] - 0s 3ms/step - loss: 5083.0371 - val_loss: 3191.7368
Epoch 3/300
25/25 [==============================] - 0s 4ms/step - loss: 2112.8142 - val_loss: 1191.2080
Epoch 4/300
25/25 [==============================] - 0s 3ms/step - loss: 670.3354 - val_loss: 89.8586
Epoch 5/300
25/25 [==============================] - 0s 3ms/step - loss: 54.6025 - val_loss: 44.1877
Epoch 6/300
25/25 [==============================] - 0s 3ms/step - loss: 37.3965 - val_loss: 89.6563
Epoch 7/300
25/25 [==============================] - 0s 3ms/step - loss: 34.3256 - val_loss: 42.2236
Epoch 8/300
25/25 [==============================] - 0s 3ms/step - loss: 28.1542 - val_loss: 89.6841
Epoch 9/300
25/25 [==============================] - 0s 3ms/step - loss: 29.3743 - val_loss: 73.8181
Epoch 10/300
25/25 [==============================] - 0s 4ms/step - loss: 29.

In [12]:
#check for improved results
y_pred = model2.predict(X_test)
print(f'The mean squared error for the test data is {np.round(mean_squared_error(y_test,y_pred),4)}')

10/10 [==============================] - 0s 2ms/step
The mean squared error for the test data is 27.4649


The model was improved by changing the optimizer, increasing the epochs, and decreasing the neurons

# PyTorch

In [15]:
X_train_pt = torch.tensor(X_train.astype(np.float32))
y_train_pt = torch.tensor(y_train.values.astype(np.float32).reshape(-1,1))

In [18]:
input_size = X_train_pt.shape[1]
output_size = y_train_pt.shape[1]
hidden_size = 10
class LinearRegressionModel(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(LinearRegressionModel, self).__init__()
        self.hidden = torch.nn.Linear(input_size, hidden_size)
        self.predict = torch.nn.Linear(hidden_size, output_size)
    def forward(self, x):
        x = F.relu(self.hidden(x))
        y_pred = self.predict(x)
        return y_pred

In [22]:
model = LinearRegressionModel(input_size, hidden_size, output_size)
l = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=.001, momentum=.9)
torch.manual_seed(42)
np.random.seed(42)

epochs=100
for x in range(epochs):
  y_pred = model(X_train_pt.requires_grad_())

  loss = l(y_pred,y_train_pt)

  optimizer.zero_grad()

  loss.backward()

  optimizer.step()

  print(f'epoch: {x}, loss:{loss.item()}')

epoch: 0, loss:15607.0478515625
epoch: 1, loss:15490.1943359375
epoch: 2, loss:15280.560546875
epoch: 3, loss:14936.306640625
epoch: 4, loss:14259.595703125
epoch: 5, loss:12753.8154296875
epoch: 6, loss:9457.931640625
epoch: 7, loss:3749.3603515625
epoch: 8, loss:631.8353881835938
epoch: 9, loss:8594.544921875
epoch: 10, loss:5634.076171875
epoch: 11, loss:308.5087890625
epoch: 12, loss:3004.26220703125
epoch: 13, loss:5448.4306640625
epoch: 14, loss:5524.0615234375
epoch: 15, loss:3479.36181640625
epoch: 16, loss:650.5983276367188
epoch: 17, loss:933.188720703125
epoch: 18, loss:3822.21044921875
epoch: 19, loss:1677.498779296875
epoch: 20, loss:115.14611053466797
epoch: 21, loss:1390.1025390625
epoch: 22, loss:2359.6328125
epoch: 23, loss:1886.3477783203125
epoch: 24, loss:559.9949340820312
epoch: 25, loss:168.09669494628906
epoch: 26, loss:1349.937255859375
epoch: 27, loss:1257.5032958984375
epoch: 28, loss:160.04269409179688
epoch: 29, loss:294.9023132324219
epoch: 30, loss:901.418

In [23]:
X_test_pt = torch.from_numpy(X_test.astype(np.float32))
y_pred = model(X_test_pt).detach().numpy()

print(f'The mean squared error is: {np.round(mean_squared_error(y_test, y_pred))}')

The mean squared error is: 23.0


In [24]:
#Change the amount of epochs and optimizer to see if we can get better results
model = LinearRegressionModel(input_size, hidden_size, output_size)
l = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=.001)
torch.manual_seed(42)
np.random.seed(42)

epochs=200
for x in range(epochs):
  y_pred = model(X_train_pt.requires_grad_())

  loss = l(y_pred,y_train_pt)

  optimizer.zero_grad()

  loss.backward()

  optimizer.step()

  print(f'epoch: {x}, loss:{loss.item()}')

epoch: 0, loss:15607.0478515625
epoch: 1, loss:15605.75390625
epoch: 2, loss:15604.4658203125
epoch: 3, loss:15603.177734375
epoch: 4, loss:15601.890625
epoch: 5, loss:15600.607421875
epoch: 6, loss:15599.32421875
epoch: 7, loss:15598.04296875
epoch: 8, loss:15596.763671875
epoch: 9, loss:15595.486328125
epoch: 10, loss:15594.2099609375
epoch: 11, loss:15592.93359375
epoch: 12, loss:15591.66015625
epoch: 13, loss:15590.3876953125
epoch: 14, loss:15589.1181640625
epoch: 15, loss:15587.8466796875
epoch: 16, loss:15586.5771484375
epoch: 17, loss:15585.3115234375
epoch: 18, loss:15584.04296875
epoch: 19, loss:15582.77734375
epoch: 20, loss:15581.5087890625
epoch: 21, loss:15580.244140625
epoch: 22, loss:15578.98046875
epoch: 23, loss:15577.7158203125
epoch: 24, loss:15576.4521484375
epoch: 25, loss:15575.189453125
epoch: 26, loss:15573.9248046875
epoch: 27, loss:15572.66015625
epoch: 28, loss:15571.396484375
epoch: 29, loss:15570.1298828125
epoch: 30, loss:15568.8603515625
epoch: 31, loss:

In [25]:
#Check to see if this is better
X_test_pt = torch.from_numpy(X_test.astype(np.float32))
y_pred = model(X_test_pt).detach().numpy()

print(f'The mean squared error is: {np.round(mean_squared_error(y_test, y_pred))}')

The mean squared error is: 15195.0


The first run through was much better and cleaner